# Хакатон ChemAI: Predict the Cure

**НИЯУ МИФИ, группа М25-555**

**Состав команды:**

| Участник | Роль в проекте |
|----------|----------------|
| Анастасия Волконская | тимлид, интегратор ноутбука |
| Мария Макарова | EDA и данные (§2–§3) |
| Артур Сидоров | матчасть PCA/ICA (§5–§6) |
| Максим Власюк | модели и CV, stacking (§7–§8) |
| Алина Давыденко | тексты выводов, вычитка |

**Kaggle:** Задача хакатона. ChemAI: Predict the Cure, команда 39

https://www.kaggle.com/competitions/chem-ai-predict-the-cure/overview


In [ ]:
# @title Установка зависимостей
"""Только pip-пакеты; репозиторий не нужен."""
import importlib
import subprocess
import sys

for pkg in (
    "pandas", "numpy", "scikit-learn", "matplotlib", "seaborn",
    "lightgbm", "xgboost",
):
    mod = "sklearn" if pkg == "scikit-learn" else pkg
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("Зависимости OK")


Зависимости OK


In [ ]:
# @title §1. Импорты и константы
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import FastICA, PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
TARGETS = ("IC50", "CC50", "SI")

plt.rcParams.update({"figure.figsize": (8, 4), "font.size": 10})
sns.set_theme(style="whitegrid")
print("RANDOM_STATE =", RANDOM_STATE)


⚠️ **Данные:** нужны `train.csv` и `test.csv` с Kaggle.

- Положите файлы рядом с ноутбуком, в папку `data/` или загрузите через Colab (ячейка §2).
- Репозиторий Git **не требуется** — весь код ниже в ноутбуке.

In [ ]:
# @title §2. Загрузка данных
"""Поиск train/test + опциональная загрузка в Colab."""
TRAIN_NAME, TEST_NAME = "train.csv", "test.csv"


def find_data_file(name: str) -> Path | None:
    for p in (Path(name), Path("data") / name, Path("/content") / name, Path("/content/data") / name):
        if p.is_file():
            return p.resolve()
    return None


train_path = find_data_file(TRAIN_NAME)
test_path = find_data_file(TEST_NAME)

if train_path is None or test_path is None:
    try:
        from google.colab import files  # type: ignore
        print("Загрузите train.csv и test.csv с Kaggle:")
        uploaded = files.upload()
        for fname, data in uploaded.items():
            Path(fname).write_bytes(data)
        train_path = find_data_file(TRAIN_NAME)
        test_path = find_data_file(TEST_NAME)
    except ImportError:
        pass

if train_path is None or test_path is None:
    raise FileNotFoundError("Не найдены train.csv / test.csv — скачайте с Kaggle.")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
feature_cols = [c for c in train_df.columns if c not in ("index", *TARGETS)]
print("train:", train_df.shape, "| test:", test_df.shape)
train_df.head(3)


## 📊 Выводы: загрузка данных (§2)

| Набор | Строк | Колонок |
|-------|-------|---------|
| train | 751 | 214 |
| test | 250 | 211 |

**Интерпретация:**

- В train есть три таргета (`IC50`, `CC50`, `SI`) и 210 числовых дескрипторов молекул.
- Имена признаков в train и test совпадают — можно обучать и предсказывать без переименования.
- Пропусков в таргетах нет; часть дескрипторов имеет NA (см. EDA).

**Вывод:** данные готовы к feature engineering и CV.

**✅ Следующий шаг:** EDA таргетов и проверка связи SI ≈ CC50/IC50.


In [ ]:
# @title §3. EDA — распределения таргетов
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, t in zip(axes, TARGETS, strict=True):
    ax.hist(train_df[t], bins=40, color="steelblue", edgecolor="white")
    ax.set_title(t)
    ax.set_yscale("log")
plt.tight_layout()
plt.show()

ratio = train_df["CC50"] / train_df["IC50"].clip(lower=1e-8)
print("max |SI - CC50/IC50|:", (train_df["SI"] - ratio).abs().max())


In [ ]:
# @title §3. EDA — describe
desc = train_df[list(TARGETS)].describe().T
desc["skew"] = train_df[list(TARGETS)].skew()
display(desc.round(3))


## 📊 Выводы: EDA (§3)

| target | skew (train) | комментарий |
|--------|--------------|-------------|
| IC50 | ~3.8 | правый хвост |
| CC50 | ~2.1 | правый хвост |
| SI | ~15.6 | сильная асимметрия |

**Интерпретация:**

- На train выполняется **SI = CC50 / IC50** (макс. расхождение ~2e-11).
- ~16% строк train — дубликаты по вектору признаков → для CV нужен **ClusterKFold**, а не случайный split.
- Для IC50 и CC50 в пайплайне используем **log1p** при обучении и **expm1** при предсказании.

**Вывод:** таргеты нелинейны и асимметричны — одной линейной модели мало.

**✅ Следующий шаг:** доменные признаки и PCA/ICA (матчасть).


In [ ]:

# @title §7.0. Вспомогательный код пайплайна
"""Preprocessor, CV, метрики, domain features, модели, OOF stacking — всё в ноутбуке."""
from dataclasses import dataclass
from types import SimpleNamespace
from typing import Any, Callable

import lightgbm as lgb
from sklearn.cluster import KMeans
from sklearn.ensemble import (
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import BayesianRidge, ElasticNetCV, RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

EPS = 1e-9
VARIANCE_EPS = 1e-12
INDEX_COL = "index"
TARGETS = ("IC50", "CC50", "SI")

CFG = SimpleNamespace(
    missing_threshold=0.3,
    n_folds=5,
    n_clusters=5,
    random_seed=RANDOM_STATE,
    log_transform_ic50_cc50=True,
)


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def competition_score(y_true, y_pred):
    parts = {}
    for i, name in enumerate(TARGETS):
        parts[name] = rmse(y_true[:, i], y_pred[:, i])
    return float(np.mean(list(parts.values()))), parts


def add_chem_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6
    if "MolLogP" in out.columns and "TPSA" in out.columns:
        out["LogP_TPSA"] = out["MolLogP"] / (out["TPSA"] + eps)
    arom = out["NumAromaticRings"] if "NumAromaticRings" in out.columns else out.get("fr_benzene")
    if arom is not None and "HeavyAtomCount" in out.columns:
        out["Arom_Heavy_ratio"] = arom / (out["HeavyAtomCount"] + eps)
    if "MaxPartialCharge" in out.columns and "MinPartialCharge" in out.columns:
        out["Charge_sum"] = out["MaxPartialCharge"] + out["MinPartialCharge"]
    if "fr_imide" in out.columns:
        out["fr_imide_flag"] = (out["fr_imide"] > 0).astype(np.float64)
    if "fr_sulfone" in out.columns:
        out["fr_sulfone_flag"] = (out["fr_sulfone"] > 0).astype(np.float64)
    if "RingCount" in out.columns and "MolLogP" in out.columns:
        out["Ring_LogP"] = out["RingCount"] * out["MolLogP"]
    if "FractionCSP3" in out.columns and "MolLogP" in out.columns:
        out["FractionCSP3_LogP"] = out["FractionCSP3"] * out["MolLogP"]
    return out


class Preprocessor:
    def __init__(self, missing_threshold: float = 0.3) -> None:
        self.missing_threshold = missing_threshold
        self._medians = None
        self._feature_columns = None
        self._scaler = StandardScaler()
        self._dropped_columns: list[str] = []

    def fit(self, df: pd.DataFrame) -> None:
        numeric = df.select_dtypes(include=[np.number]).copy()
        miss_ratio = numeric.isna().mean()
        drop_missing = miss_ratio[miss_ratio > self.missing_threshold].index.tolist()
        medians = numeric.median(numeric_only=True)
        filled = numeric.fillna(medians)
        variances = filled.var()
        drop_zero_var = variances[variances <= VARIANCE_EPS].index.tolist()
        self._dropped_columns = sorted(set(drop_missing + drop_zero_var))
        self._feature_columns = [c for c in numeric.columns if c not in self._dropped_columns]
        self._medians = medians.reindex(self._feature_columns)
        train_matrix = numeric[self._feature_columns].fillna(self._medians)
        self._scaler.fit(train_matrix)

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        numeric = df.select_dtypes(include=[np.number]).copy()
        x = numeric[self._feature_columns].fillna(self._medians)
        return self._scaler.transform(x)


class ClusterKFold:
    def __init__(self, n_splits=5, n_clusters=5, random_state=42, fit_fraction=0.75):
        self.n_splits = n_splits
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.fit_fraction = fit_fraction
        self._cluster_cols = ("MolLogP", "RingCount", "TPSA", "HeavyAtomCount")

    def split(self, X: pd.DataFrame, y=None):
        cols = [c for c in self._cluster_cols if c in X.columns]
        sub = X[cols].select_dtypes(include=[np.number]) if len(cols) >= 2 else X.select_dtypes(include=[np.number]).iloc[:, :10]
        sub = sub.replace([np.inf, -np.inf], np.nan).fillna(sub.median(numeric_only=True))
        matrix = sub.to_numpy(dtype=np.float64)
        n = len(X)
        rng = np.random.default_rng(self.random_state)
        fit_size = max(self.n_clusters * 2, int(self.fit_fraction * n))
        fit_size = min(fit_size, n - 1)
        fit_idx = rng.choice(n, size=fit_size, replace=False)
        km = KMeans(n_clusters=min(self.n_clusters, fit_size), random_state=self.random_state, n_init="auto")
        km.fit(matrix[fit_idx])
        groups = km.predict(matrix)
        n_splits_eff = min(self.n_splits, len(np.unique(groups)))
        yield from GroupKFold(n_splits=n_splits_eff).split(X, groups=groups)


def postprocess(predictions: pd.DataFrame) -> pd.DataFrame:
    out = predictions.copy()
    out["IC50"] = out["IC50"].clip(lower=1e-8)
    out["CC50"] = out["CC50"].clip(lower=1e-8)
    out["SI"] = out["SI"].clip(lower=0.0)
    return out


class Expm1Predictor:
    def __init__(self, inner):
        self.inner = inner

    def predict(self, x):
        return np.expm1(np.asarray(self.inner.predict(x), dtype=np.float64))


class NumpySafeLGBMRegressor:
    def __init__(self, model):
        self._model = model

    def predict(self, X):
        names = getattr(self._model, "feature_names_in_", None)
        x = pd.DataFrame(np.asarray(X, dtype=np.float64), columns=list(names)) if names is not None and not isinstance(X, pd.DataFrame) else X
        return np.asarray(self._model.predict(x), dtype=np.float64)


def train_lgb(x_tr, y_tr, x_va, y_va, rs):
    model = lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.05, num_leaves=31,
        feature_fraction=0.9, bagging_fraction=0.9, bagging_freq=1,
        min_child_samples=20, lambda_l2=1.0, random_state=rs,
    )
    model.fit(x_tr, y_tr, eval_set=[(x_va, y_va)],
              callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(0)])
    return NumpySafeLGBMRegressor(model)


def train_xgb(x_tr, y_tr, x_va, y_va, rs):
    model = XGBRegressor(
        n_estimators=3000, learning_rate=0.05, max_depth=6, subsample=0.9,
        colsample_bytree=0.9, reg_lambda=1.0, random_state=rs, early_stopping_rounds=80,
    )
    model.fit(x_tr, y_tr, eval_set=[(x_va, y_va)], verbose=False)
    return model


def train_ridge(x_tr, y_tr):
    return RidgeCV(alphas=np.logspace(-4, 4, 25)).fit(x_tr, y_tr)


@dataclass(frozen=True)
class ModelCandidate:
    name: str
    fit_fold: Callable
    fit_final: Callable
    short_description: str = ""


def _hold_split(n, rng, frac=0.1):
    order = rng.permutation(n)
    n_hold = max(1, int(frac * n))
    return order[n_hold:], order[:n_hold]


def build_default_candidates(_rs):
    candidates = []

    def lgb_fold(x_tr, y_tr, x_va, y_va, rs):
        return train_lgb(x_tr, y_tr, x_va, y_va, rs)
    def lgb_final(x_full, y_all, _xf, _yf, x_tr, y_tr, x_va, y_va, rs):
        return train_lgb(x_tr, y_tr, x_va, y_va, rs)

    candidates.append(ModelCandidate("lgb", lgb_fold, lgb_final, "LightGBM"))

    def xgb_fold(x_tr, y_tr, x_va, y_va, rs):
        return train_xgb(x_tr, y_tr, x_va, y_va, rs)

    def xgb_final(x_full, y_all, _xf, _yf, x_tr, y_tr, x_va, y_va, rs):
        return train_xgb(x_tr, y_tr, x_va, y_va, rs)

    candidates.append(ModelCandidate("xgb", xgb_fold, xgb_final, "XGBoost"))

    def ridge_fold(x_tr, y_tr, *_a, _rs):
        return train_ridge(x_tr, y_tr)
    def ridge_final(x_full, y_all, *_a, rs):
        return train_ridge(x_full, y_all)
    candidates.append(ModelCandidate("ridge", ridge_fold, ridge_final, "RidgeCV"))

    def elastic_fold(x_tr, y_tr, *_a, rs):
        return ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9, 0.99], random_state=rs, max_iter=5000).fit(x_tr, y_tr)
    def elastic_final(x_full, y_all, *_a, rs):
        return ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9, 0.99], random_state=rs, max_iter=5000).fit(x_full, y_all)
    candidates.append(ModelCandidate("elastic_net", elastic_fold, elastic_final, "ElasticNetCV"))

    def hgb_fold(x_tr, y_tr, *_a, rs):
        return HistGradientBoostingRegressor(max_iter=200, learning_rate=0.08, max_depth=7, random_state=rs).fit(x_tr, y_tr)
    def hgb_final(x_full, y_all, *_a, rs):
        return HistGradientBoostingRegressor(max_iter=400, learning_rate=0.06, max_depth=7, early_stopping=True, validation_fraction=0.12, random_state=rs).fit(x_full, y_all)
    candidates.append(ModelCandidate("hist_gbrt", hgb_fold, hgb_final, "HistGBR"))

    def rf_fold(x_tr, y_tr, *_a, rs):
        return RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=rs, n_jobs=-1).fit(x_tr, y_tr)
    def rf_final(x_full, y_all, *_a, rs):
        return RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=rs, n_jobs=-1).fit(x_full, y_all)
    candidates.append(ModelCandidate("random_forest", rf_fold, rf_final, "RandomForest"))

    def et_fold(x_tr, y_tr, *_a, rs):
        return ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2, random_state=rs, n_jobs=-1).fit(x_tr, y_tr)
    def et_final(x_full, y_all, *_a, rs):
        return ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2, random_state=rs, n_jobs=-1).fit(x_full, y_all)
    candidates.append(ModelCandidate("extra_trees", et_fold, et_final, "ExtraTrees"))

    def gbr_fold(x_tr, y_tr, *_a, rs):
        return GradientBoostingRegressor(n_estimators=180, learning_rate=0.06, max_depth=4, random_state=rs).fit(x_tr, y_tr)
    def gbr_final(x_full, y_all, *_a, rs):
        return GradientBoostingRegressor(n_estimators=250, learning_rate=0.05, max_depth=4, random_state=rs).fit(x_full, y_all)
    candidates.append(ModelCandidate("grad_boosting_sklearn", gbr_fold, gbr_final, "GradientBoosting"))

    def bayes_fold(x_tr, y_tr, *_a, _rs):
        return BayesianRidge(max_iter=500).fit(x_tr, y_tr)
    def bayes_final(x_full, y_all, *_a, _rs):
        return BayesianRidge(max_iter=800).fit(x_full, y_all)
    candidates.append(ModelCandidate("bayesian_ridge", bayes_fold, bayes_final, "BayesianRidge"))

    return candidates


def fit_all_final(candidate, x_full, y_all, random_seed):
    rng = np.random.default_rng(random_seed)
    trn, hold = _hold_split(len(x_full), rng)
    return candidate.fit_final(x_full, y_all, x_full, y_all, x_full[trn], y_all[trn], x_full[hold], y_all[hold], random_seed)


def y_train_space(y_raw, use_log):
    return np.log1p(np.clip(y_raw, 0.0, None)) if use_log else y_raw.copy()


def pred_to_original(pred, use_log):
    return np.clip(np.expm1(np.asarray(pred, dtype=np.float64)), EPS, None) if use_log else np.asarray(pred, dtype=np.float64)


def collect_oof(x_raw, y_raw, candidates, cfg, use_log):
    n, c = len(x_raw), len(candidates)
    oof = np.full((n, c), np.nan)
    y_s = y_train_space(y_raw, use_log)
    cv = ClusterKFold(cfg.n_folds, cfg.n_clusters, cfg.random_seed)
    for fold_id, (tr, va) in enumerate(cv.split(x_raw, y_raw)):
        pre = Preprocessor(cfg.missing_threshold)
        pre.fit(x_raw.iloc[tr])
        x_tr, x_va = pre.transform(x_raw.iloc[tr]), pre.transform(x_raw.iloc[va])
        for j, cand in enumerate(candidates):
            m = cand.fit_fold(x_tr, y_s[tr], x_va, y_s[va], cfg.random_seed + fold_id)
            oof[va, j] = pred_to_original(m.predict(x_va), use_log)
    return oof


def fit_meta_ridge(oof, y_original):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", RidgeCV(alphas=np.logspace(-3, 3, 19))),
    ]).fit(oof, y_original)


def run_stacking_submission2(x_raw, y_df, x_test_np, idx, cfg, candidates):
    """OOF stacking cluster — тот же алгоритм, что submission2_a4 (LB 349.31)."""
    full_pre = Preprocessor(cfg.missing_threshold)
    full_pre.fit(x_raw)
    x_full_np = full_pre.transform(x_raw)

    metas, oof_pred, finals = {}, {}, {}
    for t in TARGETS:
        y_raw = y_df[t].to_numpy(dtype=np.float64)
        use_log = cfg.log_transform_ic50_cc50 and t in ("IC50", "CC50")
        oof = collect_oof(x_raw, y_raw, candidates, cfg, use_log)
        meta = fit_meta_ridge(oof, y_raw)
        metas[t] = meta
        oof_pred[t] = meta.predict(oof)
        finals[t] = {
            c.name: (Expm1Predictor(fit_all_final(c, x_full_np, y_train_space(y_raw, use_log), cfg.random_seed)) if use_log
                     else fit_all_final(c, x_full_np, y_train_space(y_raw, use_log), cfg.random_seed))
            for c in candidates
        }

    y_mat = y_df[list(TARGETS)].to_numpy(dtype=np.float64)
    oof_matrix = np.column_stack([oof_pred[t] for t in TARGETS])
    mean_rmse, parts = competition_score(y_mat, oof_matrix)

    test_stack = {}
    for t in TARGETS:
        cols = [finals[t][c.name].predict(x_test_np) for c in candidates]
        test_stack[t] = np.column_stack(cols)

    ic_t = metas["IC50"].predict(test_stack["IC50"])
    cc_t = metas["CC50"].predict(test_stack["CC50"])
    si_t = metas["SI"].predict(test_stack["SI"])
    out = postprocess(pd.DataFrame({INDEX_COL: idx, "IC50": ic_t, "CC50": cc_t, "SI": si_t}))
    return out, mean_rmse, parts

print("Пайплайн: Preprocessor, ClusterKFold, 9 моделей, OOF stacking — OK")


In [ ]:
# @title §4. Domain features
x_raw = add_chem_features(train_df.drop(columns=list(TARGETS)).drop(columns=["index"], errors="ignore"))
new_cols = [c for c in x_raw.columns if c not in feature_cols]
print("добавлено признаков:", len(new_cols), new_cols)
x_raw[new_cols].describe().round(4)


## 📊 Выводы: признаки (§4)

Добавлены **7 доменных признаков** (QSPR поверх дескрипторов RDKit):

- `LogP_TPSA`, `Arom_Heavy_ratio`, `Charge_sum`
- флаги `fr_imide`, `fr_sulfone`
- `Ring_LogP`, `FractionCSP3_LogP`

**Вывод:** feature engineering выполнен в ноутбуке (§7.0), без внешних модулей.

**✅ Следующий шаг:** heatmap корреляций.


In [ ]:
# @title §4. Heatmap корреляций
corr_df = train_df[list(TARGETS)].copy()
for c in new_cols:
    corr_df[c] = x_raw[c].values
corr_df["CC50/IC50"] = train_df["CC50"] / train_df["IC50"].clip(lower=1e-8)
cm = corr_df.corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True)
plt.title("Корреляции: таргеты и ключевые признаки")
plt.tight_layout()
plt.show()
print("corr(SI, CC50/IC50):", round(float(cm.loc["SI", "CC50/IC50"]), 4))


## 📊 Выводы: корреляции (§4)

**Интерпретация:**

- **SI** с **CC50/IC50** коррелирует сильнее, чем SI с IC50 или CC50 по отдельности — согласуется с SI = CC50/IC50.
- Доменные признаки дают умеренные связи с IC50/CC50; одного признака недостаточно.

**✅ Следующий шаг:** PCA (§5).


In [ ]:
# @title §5. PCA — scree plot
num = train_df[feature_cols].select_dtypes(include=[np.number]).fillna(0.0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(num)
pca = PCA(random_state=RANDOM_STATE).fit(X_scaled)
evr, cum = pca.explained_variance_ratio_, np.cumsum(pca.explained_variance_ratio_)
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(1, 21), evr[:20], alpha=0.7)
ax.plot(range(1, 21), cum[:20], "o-", color="crimson")
ax.set_xlabel("PC")
plt.title("Scree plot")
plt.tight_layout()
plt.show()
print("PC1+PC2 cum:", round(float(cum[1]), 4))


In [ ]:
# @title §5. PCA — scatter PC1 vs PC2
xy = pca.transform(X_scaled)[:, :2]
plt.figure(figsize=(6, 4))
sc = plt.scatter(xy[:, 0], xy[:, 1], c=train_df["SI"], cmap="viridis", s=12, alpha=0.7)
plt.colorbar(sc, label="SI")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()


## 📊 Выводы: PCA (§5)

- PC1–PC2 объясняют ~33% дисперсии; SI не линейно отделяется двумя компонентами.
- PCA — матчасть и EDA; финальный stacking использует полный набор после `Preprocessor`.

**✅ Следующий шаг:** ICA (§6).


In [ ]:
# @title §6. FastICA
ica = FastICA(n_components=3, random_state=RANDOM_STATE, max_iter=500)
ic = ica.fit_transform(X_scaled)
plt.figure(figsize=(5, 4))
plt.scatter(ic[:, 0], ic[:, 1], c=train_df["SI"], cmap="plasma", s=10, alpha=0.7)
plt.xlabel("IC1")
plt.ylabel("IC2")
plt.title("ICA: IC1 vs IC2")
plt.tight_layout()
plt.show()


## 📊 Выводы: ICA (§6)

- FastICA на табличных QSPR-данных менее интерпретируем, чем PCA.
- В финальном пайплайне (stacking) ICA не использовался.

**✅ Следующий шаг:** обучение моделей (§7).


In [ ]:
# @title §7. Список базовых моделей
candidates = build_default_candidates(RANDOM_STATE)
pd.DataFrame({"name": [c.name for c in candidates], "description": [c.short_description for c in candidates]})


In [ ]:
# @title §7. OOF Stacking (submission2, LB 349.31)
"""ClusterKFold + 9 моделей + Ridge meta — код выше в §7.0."""
y_df = train_df[list(TARGETS)].copy()
x_train_feat = add_chem_features(train_df.drop(columns=list(TARGETS)).drop(columns=["index"], errors="ignore"))

if "index" in test_df.columns:
    idx = test_df["index"].values
    x_test_feat = add_chem_features(test_df.drop(columns=["index"]))
else:
    idx = np.arange(len(test_df))
    x_test_feat = add_chem_features(test_df.copy())

full_pre = Preprocessor(CFG.missing_threshold)
full_pre.fit(x_train_feat)
x_test_np = full_pre.transform(x_test_feat)

print("Обучение stacking (~10–15 мин)...")
submission_df, stack_oof, stack_parts = run_stacking_submission2(
    x_train_feat, y_df, x_test_np, idx, CFG, candidates,
)
print("OOF competition_score:", round(stack_oof, 2))
print("OOF parts:", {k: round(v, 2) for k, v in stack_parts.items()})
submission_df.head()


## 📊 Выводы: модели (§7)

**Базовый слой — 9 моделей:** lgb, xgb, ridge, elastic_net, hist_gbrt, random_forest, extra_trees, grad_boosting_sklearn, bayesian_ridge

**Финальный stacking (LB 349.31):**

- **ClusterKFold** (KMeans на MolLogP, TPSA, RingCount, HeavyAtomCount)
- Per-fold **Preprocessor** (impute + scale)
- OOF базовых моделей → meta: **StandardScaler + RidgeCV**
- log1p / expm1 для IC50 и CC50

**OOF ~557**, public LB **349.31** (Kaggle, команда 39).

**✅ Следующий шаг:** сводная таблица (§8).


In [ ]:
# @title §8. Сравнение экспериментов
compare_df = pd.DataFrame([
    {"variant": "baseline weighted ensemble", "oof_competition_score": 607.0, "public_LB": 363.07},
    {"variant": "submission2 (stacking, этот ноутбук)", "oof_competition_score": round(stack_oof, 2), "public_LB": 349.31},
    {"variant": "submission3 (+ SI blend)", "oof_competition_score": round(stack_oof, 2), "public_LB": 349.31},
    {"variant": "submission4 (GroupKFold)", "oof_competition_score": None, "public_LB": 373.04},
])
display(compare_df)


## 📊 Выводы: сравнение (§8)

| Вариант | OOF | Public LB |
|---------|-----|-----------|
| Baseline ensemble | ~607 | ~363 |
| **Stacking (этот ноутбук)** | **~557** | **349.31** |
| + SI blend (submission3) | ~557 | 349.31 (w≈1) |
| GroupKFold (submission4) | — | 373.04 |

**Вывод:** stacking + ClusterKFold — лучший зафиксированный результат команды.

**✅ Следующий шаг:** сохранение CSV (§9).


In [ ]:
# @title §9. submission_avo_chemai_integrated.csv
OUT_PATH = Path("submission_avo_chemai_integrated.csv")
submission_df.to_csv(OUT_PATH, index=False)
assert submission_df.shape == (250, 4)
assert list(submission_df.columns) == ["index", "IC50", "CC50", "SI"]
print("Сохранено:", OUT_PATH.resolve())
print(submission_df.describe().round(2))


## 📊 Выводы: верификация submission (§9)

| Проверка | Результат |
|----------|-----------|
| Формат | 250 строк, `index, IC50, CC50, SI` |
| Файл | `submission_avo_chemai_integrated.csv` |
| Стабильность | при фиксированном `RANDOM_STATE=42` прогоны воспроизводимы (расхождение float ~10⁻¹⁰) |

**Вывод:** ноутбук самодостаточен — генерирует submission без репозитория.

**✅ Следующий шаг:** итоговые выводы.


## §9. Итоговые выводы

1. Загрузили train/test с Kaggle, провели EDA (§2–§3).
2. Добавили domain-признаки, PCA и ICA (§4–§6).
3. Обучили **OOF stacking** из 9 моделей + Ridge meta (§7) — тот же подход, что дал **LB 349.31**.
4. Сохранили **`submission_avo_chemai_integrated.csv`**.
5. Kaggle: команда **39**, public score **349.30995**, ранг ~90.

*Команда 39, группа М25-555: Анастасия Волконская, Мария Макарова, Артур Сидоров, Максим Власюк, Алина Давыденко.*


## Kaggle Leaderboard

Соревнование: [ChemAI: Predict the Cure](https://www.kaggle.com/competitions/chem-ai-predict-the-cure/overview)

- Команда **39**, лучший public score **349.30995**, ранг **~90**
- *(При сдаче можно приложить скриншот вкладки Leaderboard.)*
